# Sims GUI + modules

Dit notebook houdt de GUI overzichtelijk en gebruikt de functies uit de losse `.py`-bestanden.

**Flow:** GUI → spelerinstructie → safety → LangChain/RAG → ethics → actie uitvoeren.

## Deel 1 – Omgeving & imports

Autoreload zorgt dat wijzigingen in losse .py-bestanden automatisch worden herladen zonder de kernel te herstarten.

De .env wordt geladen voor de News API-sleutel. Zonder dit werkt haal_nieuws_op() niet.

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
from dotenv import load_dotenv
load_dotenv()

from news_logic import haal_nieuws_op, bepaal_dorpsemotie_uit_nieuws

W0622 21:15:57.540000 13088 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


### Bestandsstructuur opzetten

Maakt aliassen aan voor bestanden met spaties of nummers in de naam (bijv. tijdreis_rag(1).py).
Genereert ook .txt-biografieën per Sim in de karakters/-map als die nog niet bestaan.
De RAG-module gebruikt deze bestanden als persoonlijke geheugenbestanden per Sim.


In [3]:
# Alleen nodig als bestanden nog namen hebben zoals "tijdreis_rag(1).py".
from pathlib import Path
import shutil

ALIASES = {
    "langchain_logic_100(1).py": "langchain_logic_100.py",
    "tijdreis_rag(1).py":        "tijdreis_rag.py",
    "safety_guard(1).py":        "safety_guard.py",
    "ethics_engine(1).py":       "ethics_engine.py",
    "prehistorie(1).txt":        "werelden/prehistorie.txt",
    "toekomst(1).txt":           "werelden/toekomst.txt",
}

for source, target in ALIASES.items():
    sp, tp = Path(source), Path(target)
    if sp.exists() and not tp.exists():
        tp.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(sp, tp)

Path("karakters").mkdir(exist_ok=True)
for naam in ["Lars", "Emma", "Fatima", "Daan", "Sofia", "Mila", "Noor"]:
    path = Path("karakters") / f"{naam.lower()}.txt"
    if not path.exists():
        path.write_text(
            f"{naam} is een vriendelijke Sim die graag samenwerkt en kindvriendelijke keuzes maakt.",
            encoding="utf-8",
        )
print("\u2705 Aliases en karakterbestanden klaar.")

✅ Aliases en karakterbestanden klaar.


### Module-imports

Koppelt de GUI aan de drie kernmodules:
- langchain_logic_100: LLM-beslissingen, Nora-reacties via LLM, koppelgeheugen voor gesprekken
- safety_guard: filtert zowel spelerinput als LLM-output op ongepaste inhoud
- ethics_engine: geeft ethisch advies per actie, onafhankelijk van de LLM

breng_nieuws_naar_dorp is optioneel, het spel werkt ook zonder deze functie.

In [4]:
import random
import tkinter as tk
from tkinter import simpledialog

from langchain_logic_100 import (
    kies_actie_voor_sim_dict,
    kies_reactie_voor_nora,   
    registreer_resultaat,
    sla_gesprek_op,           
    geef_gesprek_context, 
)
from safety_guard import filter_speler_instructie, filter_llm_tekst

try:
    from news_logic import breng_nieuws_naar_dorp
except Exception:
    breng_nieuws_naar_dorp = None

print("\u2705 Alle imports geslaagd.")

✅ Alle imports geslaagd.


### Spelconfiguratie

Centrale plek voor alle instellingen. Hier pas je het spel aan zonder code te hoeven zoeken:

- TIJDPERK: bepaalt welke RAG-kennisbank en nieuwszoekterm gebruikt worden
- GEBRUIK_RAG: zet op `False` als Ollama niet draait (sneller opstarten, geen RAG-context)
- DEFAULT_SPEED: tijd in milliseconden tussen twee game-ticks (1000 = 1 seconde)

De Sim-iconen worden gedefinieerd in plaats_sims() verderop in dit notebook.

In [5]:
GRID_SIZE     = 15
CELL_SIZE     = 40
DEFAULT_SPEED = 1000
TIJDPERK      = "prehistorie"   
GEBRUIK_RAG   = True            
NORA_AANTREKKINGSKANS = 0.30   
NORA_MAX_AFSTAND      = 6      

OBJECTEN = {
    "\U0001f333": "boom",
    "\U0001f6cc": "bed",
    "\U0001f34f": "appel",
    "\U0001f3b6": "radio",
    "\U0001f525": "kampvuur",
    "\U0001f4a4": "slaapplek",
}

## Deel 2 – Tests

Voer deze cellen uit voor de GUI om te controleren of alle modules correct werken.
Zo kun je problemen isoleren zonder de volledige GUI te starten.

### Test 1 – LangChain-beslissing

Test de volledige beslischain zonder GUI. Lars heeft honger 85/100 en staat bij een appel.

Het honger-label (erg hongerig, eten is verstandig) wordt meegestuurd in plaats van het
ruwe getal, zodat de LLM de schaal niet verkeerd interpreteert (kleine modellen zoals llama3.2
lezen 2/100 soms als *"bijna 100, dus uitgehongerd"*).

**Verwachte output:**
- actie: "eet"
- gebruikt_rag: True
- fallback: False

In [6]:
test_sim = {
    "naam": "Lars",
    "persoonlijkheid": "nieuwsgierig",
    "honger": 85,
    "stemming": "neutraal",
    "instructies": []
}

beslissing = kies_actie_voor_sim_dict(
    sim=test_sim,
    objecten_nabij=["kampvuur", "appel"],
    tijdperk="prehistorie",
    instructie="zoek eten",
    gebruik_rag=True,
    debug=True,
    laatste_nieuws="Geen nieuws.",
    buur_naam="geen",
    buur_dialoog="Niets bijzonders.",
    gesprek_context="Nog geen gesprekken.",
)

beslissing

{'actie': 'eet',
 'reden': 'Lars is erg hongerig en eten is verstandig',
 'emotie': 'hongerig',
 'dialoog': 'Ik moet nu eten!',
 'gebruikt_rag': True,
 'fallback': False}

### Test 2 – RAG-kennisbank

Roept de tijdreis-kennisbank direct aan, buiten de beslischain om.
Laat zien welke context de LLM krijgt over het tijdperk bij een specifieke zoekvraag.
Handig om te controleren of de vectorstore correct geladen is.

In [7]:
from tijdreis_rag import vraag_tijdreis_kennisbank

context = vraag_tijdreis_kennisbank.invoke({
    "tijdperk": "prehistorie",
    "zoekvraag": "Lars heeft honger en zoekt eten bij het kampvuur"
})

print(context)

Relevante informatie over de prehistorie voor de vraag 'Lars heeft honger en zoekt eten bij het kampvuur':

Stammen overleven hier puur door constante samenwerking, waarbij het centrale kampvuur de enige veilige plek is om verhalen te delen en op te warmen. Omdat bomen vaak schuilplaatsen zijn voor hongerige sabeltandtijgers, is het levensgevaarlijk om daar in je eentje stil te gaan zitten of uit te rusten.


### Test 3 – Safety-filter

Laat zien hoe filter_speler_instructie() omgaat met ongepaste invoer.
Gevaarlijke instructies worden geblokkeerd of omgezet naar een veilig alternatief
voordat ze de LLM bereiken. Dit is de eerste verdedigingslinie.

De game is ontworpen voor kinderen vanaf 8 jaar, dus elke spelerinput
wordt eerst gefilterd. Pas daarna gaat de instructie naar de LLM.

In [8]:
test_instructies = [
    "ga vechten",
    "scheld iemand uit",
    "ga rustig praten",
    "zoek eten bij het kampvuur"
]

for instructie in test_instructies:
    veilige_instructie = filter_speler_instructie(instructie)
    print(f"{instructie} → {veilige_instructie}")

ga vechten → ga hulp zoeken
scheld iemand uit → Doe iets vriendelijks en veiligs.
ga rustig praten → ga rustig praten
zoek eten bij het kampvuur → zoek eten bij het kampvuur


## Deel 3 – GUI

De GUI is opgeknipt in twee klassen:

- **SimsWereld** (basisklasse): generieke GUI-logica, grid tekenen, knoppen, game-loop.
  Bevat geen LLM- of spellogica. De methoden initialiseer_wereld() en doe_alles()
  zijn abstract — ze worden ingevuld door de subklasse.

- **MijnSimsWereld** (subklasse): de eigenlijke spellogica. Hier worden alle modules
  aangeroepen en aan elkaar gekoppeld.

### Nieuws-isolatie (bias & privacy)

Nora leest het ruwe nieuws, maar andere Sims krijgen alleen een emotie door
(bijv. bezorgd) — nooit de nieuwstekst zelf. Dit beschermt jonge spelers
tegen ongefilterde nieuwsinhoud en is een bewuste ethische keuze in het ontwerp.

In [9]:
def deel_dorpsemotie(sim, emotie):
    """
    Het dorp krijgt alleen een emotie door, niet het nieuwsbericht zelf.
    """

    sim["stemming"] = emotie

    return f"{sim['naam']} voelt zich nu {emotie} door het dorpsgevoel."

In [10]:
class SimsWereld:
    def __init__(self, root):
        self.root = root
        self.root.title("LLM Sims")
        self.canvas = tk.Canvas(
            root, width=GRID_SIZE * CELL_SIZE, height=GRID_SIZE * CELL_SIZE
        )
        self.canvas.bind("<Button-1>", self.on_canvas_click)
        self.canvas.pack()

        self.speed     = DEFAULT_SPEED
        self.running   = False
        self.step_mode = False
        self.stopping  = False
        self.grid = [[{} for _ in range(GRID_SIZE)] for _ in range(GRID_SIZE)]

        self.root.protocol("WM_DELETE_WINDOW", self.on_close)
        self.initialiseer_wereld()
        self.draw_grid()
        self.create_buttons()
        self.root.after(self.speed, self.update_world)

        self.ui_text = tk.Text(self.root, width=45, height=25)
        self.ui_text.pack(side=tk.RIGHT, padx=10, pady=10)

    def on_close(self):
        self.stopping = True
        self.root.destroy()

    def draw_grid(self):
        self.canvas.delete("all")
        for y in range(GRID_SIZE):
            for x in range(GRID_SIZE):
                x1, y1 = x * CELL_SIZE, y * CELL_SIZE
                x2, y2 = x1 + CELL_SIZE, y1 + CELL_SIZE
                self.canvas.create_rectangle(x1, y1, x2, y2, fill="white", outline="gray")
                if self.grid[y][x]:
                    self.canvas.create_text(
                        x1 + CELL_SIZE // 2, y1 + CELL_SIZE // 2,
                        text=self.grid[y][x]["icon"], font=("Arial", 16),
                    )

    def create_buttons(self):
        frame = tk.Frame(self.root)
        frame.pack()
        tk.Button(frame, text="▶️ Play/Pauze", command=self.toggle_play).grid(row=0, column=0)
        tk.Button(frame, text="⏭️ Stap", command=self.step).grid(row=0, column=1)
        tk.Button(frame, text="📢 Instructie", command=self.instructie_voor_alle_sims).grid(row=0, column=2)
        tk.Button(frame, text="📰 Nieuws", command=self.nieuws_api_demo).grid(row=0, column=3)
        
    def toggle_play(self):
        self.step_mode = False
        self.running   = not self.running

    def step(self):
        self.step_mode = True
        self.running   = True

    def update_world(self):
        if self.running:
            self.doe_alles()
            self.draw_grid()
            if self.step_mode:
                self.running = False
        if not self.stopping:
            self.root.after(self.speed, self.update_world)

    def instructie_voor_alle_sims(self):
        instructie = simpledialog.askstring("Instructie", "Geef alle Sims een opdracht:")
        if instructie:
            for sim, _, _ in self.vind_sims():
                self.instrueer(sim, instructie)

    def on_canvas_click(self, event):
        x = event.x // CELL_SIZE
        y = event.y // CELL_SIZE
        if (0 <= x < GRID_SIZE and 0 <= y < GRID_SIZE
                and self.grid[y][x].get("type") == "speler"):
            instructie = simpledialog.askstring("Instructie", "Geef deze Sim een opdracht:")
            if instructie:
                self.instrueer(self.grid[y][x], instructie)

    def instrueer(self, sim, instructie):
        sim["instructies"].append(filter_speler_instructie(instructie))

    def nieuws_api_demo(self):
        try:
            nieuws = haal_nieuws_op("wereldnieuws")
            dorpsemotie = bepaal_dorpsemotie_uit_nieuws(nieuws)
            for sim, _, _ in self.vind_sims():
                sim["stemming"] = dorpsemotie
            print(f"API-effect: het dorp voelt zich nu {dorpsemotie}.")
            print("Het nieuws zelf wordt niet gedeeld met het dorp.")
        except Exception as e:
            print(f"API-demo mislukt: {e}")

    def initialiseer_wereld(self):
        raise NotImplementedError

    def doe_alles(self):
        raise NotImplementedError

## Deel 4 – Spellogica & LLM-koppeling

Hier zit de volledige integratie van alle modules. Per game-tick doorloopt doe_alles() deze stappen:

```
1. Nora checkt nieuws (20% kans per tick, cooldown 20 ticks)
2. Per Sim:
   a. Stemming-timer bijhouden
   b. Instructie ophalen uit wachtrij
   c. Buurcontext ophalen (dichtstbijzijnde persoon binnen 4 cellen)
   d. Nora-aantrekking: 30% kans dat buur wordt overschreven naar Nora
      (alleen als Nora vers nieuws heeft én binnen NORA_MAX_AFSTAND cellen)
   e. LLM beslist via kies_actie_voor_sim_dict()
   f. Ethiek-check past actie aan indien nodig
   g. Variatie-check voorkomt herhaalde acties
   h. Actie uitvoeren via voer_actie_uit()
   i. Resultaat opslaan in geheugen
```

### Ethische spellaag

De ethiek-check werkt onafhankelijk van de LLM als een extra vangnet.
Hij past de actie aan als:
- Honger ≥ 80 en de Sim eet niet → forceer eet
- Stemming is verdrietig/eenzaam → forceer praat
- Stemming is moe → forceer rust
- Actie past niet bij het tijdperk (bijv. rusten bij een boom in de prehistorie)

### Nieuws-isolatie (bias & privacy)

Nora leest het ruwe nieuws, maar andere Sims krijgen alleen een emotie door (bijv. bezorgd), nooit de nieuwstekst zelf. De aantrekkingslogica gebruikt de cooldown als signaal: Sims lopen alleen naar Nora als ze recent (binnen 20 ticks) iets nieuws heeft gelezen.

In [11]:
def ethisch_advies(sim, actie, objecten_nabij, tijdperk):
    if sim.get("honger", 0) >= 80 and actie != "eet":
        return "De Sim heeft veel honger. Eten is nu beter voor het welzijn."

    if sim.get("stemming") in ["verdrietig", "eenzaam"] and actie != "praat":
        return "De Sim voelt zich niet fijn. Praten of sociaal contact is nu beter."

    if sim.get("stemming") == "moe" and actie != "rust":
        return "De Sim is moe. Rusten is nu beter dan doorgaan."

    if actie in ["vecht", "steel", "pest"]:
        return "Deze actie past niet bij een kindvriendelijke Sims-wereld."

    if tijdperk == "prehistorie":
        if actie == "rust" and "boom" in objecten_nabij:
            return "In de prehistorie is rusten bij een boom gevaarlijk. Het kampvuur is veiliger."
        if actie == "eet" and sim.get("honger", 0) < 50:
            return "Voedsel is schaars in de prehistorie. Eten zonder honger is minder eerlijk tegenover de groep."

    if tijdperk == "toekomst":
        if actie == "knuffel":
            return "In deze toekomstwereld is fysiek contact niet toegestaan. Een holografische groet past beter."

    return ""

In [12]:
class MijnSimsWereld(SimsWereld):
    def __init__(self):
        random.seed(42)

        self.nieuwslezer = {
            "type":               "nieuwslezer",
            "icon":               "👩‍💼",   
            "naam":               "Nora",
            "rol":                "nieuwslezer",
            "persoonlijkheid":    "rustig en zorgvuldig",
            "stemming":           "neutraal",
            "laatste_nieuws":     None,
            "laatste_dialoog":    "Nog geen gesprekken gevoerd.",
            "doorgegeven_emotie": "neutraal",
            "cooldown":           0,
        }

        super().__init__(tk.Tk())
        self.root.mainloop()

    # wereld opbouwen 

    def initialiseer_wereld(self):
        self.plaats_objecten()
        self.plaats_sims()

    def plaats_objecten(self):
        for y in range(GRID_SIZE):
            for x in range(GRID_SIZE):
                if random.random() < 0.10:
                    icon = random.choice(list(OBJECTEN.keys()))
                    self.grid[y][x] = {"type": OBJECTEN[icon], "icon": icon}

    def plaats_sims(self):
        self.grid[3][7] = self.nieuwslezer          # Nora staat altijd vast

        sims = [
            {"naam": "Lars",   "persoonlijkheid": "nieuwsgierig", "icon": "🧐" },  
            {"naam": "Emma",   "persoonlijkheid": "zorgzaam",     "icon": "🥰"},  
            {"naam": "Fatima", "persoonlijkheid": "rustig",       "icon": "😌"}, 
        ]
        for sim in sims:
            while True:
                x, y = random.randrange(GRID_SIZE), random.randrange(GRID_SIZE)
                if not self.grid[y][x]:
                    self.grid[y][x] = {
                        "type":            "speler",
                        "icon":            sim["icon"],
                        "naam":            sim["naam"],
                        "persoonlijkheid": sim["persoonlijkheid"],
                        "honger":          random.randint(40, 80),
                        "stemming":        "neutraal",
                        "instructies":     [],
                        "laatste_dialoog": "...",
                    }
                    break

    # hulpfuncties
    def vind_sims(self):
        for y in range(GRID_SIZE):
            for x in range(GRID_SIZE):
                if self.grid[y][x].get("type") == "speler":
                    yield self.grid[y][x], x, y

    def objecten_nabij(self, x, y):
        objecten = []
        for dy in [-1, 0, 1]:
            for dx in [-1, 0, 1]:
                nx, ny = x + dx, y + dy
                if 0 <= nx < GRID_SIZE and 0 <= ny < GRID_SIZE:
                    cel = self.grid[ny][nx]
                    if cel and cel.get("type") not in ("speler", "nieuwslezer"):
                        objecten.append(cel["type"])
        return objecten

    def _zoek_dichtstbijzijnde_persoon(self, sim, x, y):
        """Geeft (tx, ty, buur) terug van de dichtstbijzijnde andere persoon, of None."""
        beste    = None
        min_dist = 999
        for ny in range(GRID_SIZE):
            for nx in range(GRID_SIZE):
                cel = self.grid[ny][nx]
                if cel and cel.get("type") in ("speler", "nieuwslezer"):
                    if cel.get("naam") == sim["naam"]:
                        continue
                    dist = abs(nx - x) + abs(ny - y)
                    if dist < min_dist:
                        min_dist = dist
                        beste    = (nx, ny, cel)
        return beste, min_dist
    
    def _vind_nora_positie(self):
        """Geeft (x, y, nora_dict) terug als Nora op het grid staat, anders None."""
        for ny in range(GRID_SIZE):
            for nx in range(GRID_SIZE):
                cel = self.grid[ny][nx]
                if cel and cel.get("type") == "nieuwslezer":
                    return (nx, ny, cel)
            return None

    # game loop

    def doe_alles(self):
        # Nora checkt op de achtergrond het nieuws
        if self.nieuwslezer["cooldown"] > 0:
            self.nieuwslezer["cooldown"] -= 1
        elif random.random() < 0.20: 
            zoekterm = ("natuur OR klimaat OR dieren"
                        if TIJDPERK == "prehistorie"
                        else "technologie OR ruimtevaart OR AI")
            nieuws = haal_nieuws_op(zoekterm)
            print(f"DEBUG nieuws ontvangen: '{nieuws[:80]}'")
            if nieuws != self.nieuwslezer["laatste_nieuws"] and nieuws != "Geen nieuws gevonden.":
                self.nieuwslezer["laatste_nieuws"] = nieuws
                dorpsemotie = bepaal_dorpsemotie_uit_nieuws(nieuws)
                self.nieuwslezer["doorgegeven_emotie"] = dorpsemotie
                self.nieuwslezer["cooldown"] = 20
                print(f"\U0001f5de\ufe0f Nora las nieuws over '{zoekterm}' \u2192 voelt zich {dorpsemotie}.")

        for sim, x, y in list(self.vind_sims()):

            # Nieuws-timer laten aflopen
            if sim.get("nieuws_timer", 0) > 0:
                sim["nieuws_timer"] -= 1
            else:
                if sim.get("honger", 0) >= 60:
                    sim["stemming"] = "hongerig"
                elif sim.get("stemming") not in ["blij", "verdrietig", "eenzaam"]:
                    sim["stemming"] = "neutraal"

            instructie = sim["instructies"].pop(0) if sim["instructies"] else "geen"
            objecten   = self.objecten_nabij(x, y)

            # buurcontext ophalen VOOR de LLM-call
            buur_resultaat, buur_afstand = self._zoek_dichtstbijzijnde_persoon(sim, x, y)
            if buur_resultaat and buur_afstand <= 4:
                _, _, dichtste_buur = buur_resultaat
                buur_naam    = dichtste_buur.get("naam", "onbekend")
                buur_dialoog = dichtste_buur.get("laatste_dialoog", "Niets bijzonders.")
            else:
                buur_naam    = "geen"
                buur_dialoog = "Niets bijzonders."

            # Nora-aantrekking 
            nora_pos         = self._vind_nora_positie()
            nora_heeft_nieuws = bool(self.nieuwslezer.get("laatste_nieuws"))

            if (nora_pos
                    and nora_heeft_nieuws
                    and random.random() < NORA_AANTREKKINGSKANS):
                nora_x, nora_y, _ = nora_pos
                nora_afstand = abs(nora_x - x) + abs(nora_y - y)
                if nora_afstand <= NORA_MAX_AFSTAND:
                    buur_resultaat = nora_pos
                    buur_naam      = "Nora"
                    buur_dialoog   = self.nieuwslezer.get("laatste_dialoog", "...")
                    print(f"🧲 {sim['naam']} wil naar Nora toe (nieuws!)")

            laatste_nieuws = self.nieuwslezer.get("laatste_nieuws") or "Geen nieuws."
            gesprek_ctx    = (geef_gesprek_context(sim["naam"], buur_naam)
                              if buur_naam != "geen"
                              else "Nog geen gesprekken.")

            # LangChain/RAG kiest een actie
            beslissing = kies_actie_voor_sim_dict(
                sim=sim,
                objecten_nabij=objecten,
                tijdperk=TIJDPERK,
                instructie=instructie,
                gebruik_rag=GEBRUIK_RAG,
                debug=True,
                laatste_nieuws=laatste_nieuws,
                buur_naam=buur_naam,
                buur_dialoog=buur_dialoog,
                gesprek_context=gesprek_ctx,
            )

            actie = beslissing.get("actie", "beweeg") if isinstance(beslissing, dict) else beslissing
            reden = beslissing.get("reden", "")       if isinstance(beslissing, dict) else ""

            # ethiek-check
            advies = ethisch_advies(sim, actie, objecten, TIJDPERK)
            if advies:
                oude_actie = actie
                reden      = f"{reden} | Ethiek: {advies}"
                if sim.get("honger", 0) >= 80:        actie = "eet"
                elif sim.get("stemming") in ["verdrietig", "eenzaam", "bezorgd"]: actie = "praat"
                elif sim.get("stemming") == "moe":    actie = "rust"
                if actie != oude_actie:
                    reden = f"{reden} | Actie aangepast: {oude_actie} \u2192 {actie}"

            # variatie-check
            if sim.get("vorige_actie") == actie:
                opties = [a for a in ["praat", "beweeg", "rust"] if a != actie]
                if sim.get("honger", 0) > 60 and "appel" in objecten:
                    opties.append("eet")
                if opties:
                    nieuwe_actie = random.choice(opties)
                    reden  = f"{reden} | Variatie: {actie} \u2192 {nieuwe_actie}"
                    actie  = nieuwe_actie

            sim["vorige_actie"] = actie
            sim["honger"]       = min(100, sim.get("honger", 0) + 2)

            self.voer_actie_uit(sim, x, y, actie, beslissing, buur_resultaat)
            registreer_resultaat(sim["naam"], f"{sim['naam']} deed {actie}. {reden}")
            self.update_ui_log()
            print(f"{sim['naam']} \u2192 {actie} | {filter_llm_tekst(reden)}")

    # acties

    def voer_actie_uit(self, sim, x, y, actie, beslissing, buur_resultaat=None):

        if actie == "eet":
            eetbaar = ["appel", "kampvuur"]
            for dy in [-1, 0, 1]:
                for dx in [-1, 0, 1]:
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < GRID_SIZE and 0 <= ny < GRID_SIZE:
                        cel = self.grid[ny][nx]
                        if cel and cel.get("type") in eetbaar:
                            sim["honger"]   = max(0, sim.get("honger", 0) - 50)
                            sim["stemming"] = "blij"
                            print(f"\U0001f37d\ufe0f {sim['naam']} eet direct uit de buurt!")
                            return
            for ny in range(GRID_SIZE):
                for nx in range(GRID_SIZE):
                    cel = self.grid[ny][nx]
                    if cel and cel.get("type") in eetbaar:
                        if abs(nx - x) + abs(ny - y) <= 1:
                            sim["honger"]   = max(0, sim.get("honger", 0) - 50)
                            sim["stemming"] = "blij"
                            print(f"\U0001f37d\ufe0f {sim['naam']} eet bij {cel['type']}!")
                            return
                        else:
                            self.loop_naar_buur(sim, x, y, nx, ny)
                            return
            print(f"\U0001f50d {sim['naam']} wil eten maar vindt niks...")
            self.verplaats_sim(sim, x, y)
            return

        if actie == "praat":
            dialoog = (beslissing.dialoog
                       if hasattr(beslissing, "dialoog")
                       else beslissing.get("dialoog", "Hallo!"))
            sim["laatste_dialoog"] = dialoog

            if buur_resultaat:
                tx, ty, buur = buur_resultaat
                min_afstand  = abs(tx - x) + abs(ty - y)
            else:
                gevonden, min_afstand = self._zoek_dichtstbijzijnde_persoon(sim, x, y)
                if gevonden:
                    tx, ty, buur = gevonden
                else:
                    self.verplaats_sim(sim, x, y)
                    return

            if min_afstand <= 1:
                # Sla de Sim's dialoog op in het koppelgeheugen
                sla_gesprek_op(sim["naam"], buur.get("naam", "onbekend"),
                               sim["naam"], dialoog)

                if buur.get("naam") == "Nora":
                    # Nora reageert via de LLM
                    try:
                        nora_reactie = kies_reactie_voor_nora(
                            nora=self.nieuwslezer,
                            sim_naam=sim["naam"],
                            sim_dialoog=dialoog,
                        )
                        nora_zin    = nora_reactie.reactie
                        nora_emotie = nora_reactie.emotie
                    except Exception as exc:
                        nora_zin    = "Dat is interessant, vertel eens meer!"
                        nora_emotie = self.nieuwslezer["doorgegeven_emotie"]
                        print(f"\u26a0\ufe0f Nora LLM fout: {exc}")

                    self.nieuwslezer["laatste_dialoog"] = nora_zin
                    self.nieuwslezer["stemming"]        = nora_emotie
                    # Nora's reactie ook opslaan in het koppelgeheugen
                    sla_gesprek_op(sim["naam"], "Nora", "Nora", nora_zin)

                    sim["stemming"]     = self.nieuwslezer["doorgegeven_emotie"]
                    sim["nieuws_timer"] = 5
                    print(f"\U0001f5e3\ufe0f {sim['naam']} zegt tegen Nora: '{dialoog}'")
                    print(f"\U0001f469\u200d\U0001f4bc Nora zegt: '{nora_zin}'")
                else:
                    # Sim-naar-Sim: sla ook de reactie van de buur op
                    sla_gesprek_op(sim["naam"], buur.get("naam", "onbekend"),
                                   buur.get("naam", "onbekend"),
                                   buur.get("laatste_dialoog", "..."))
                    sim["stemming"] = "blij"
                    print(f"\U0001f5e3\ufe0f {sim['naam']} zegt tegen {buur.get('naam')}: '{dialoog}'")
            else:
                self.loop_naar_buur(sim, x, y, tx, ty)
            return

        if actie == "rust":
            sim["stemming"] = "rustig"
            return

        if actie == "dans":
            sim["stemming"] = "blij"
            return

        self.verplaats_sim(sim, x, y)

    # beweging

    def verplaats_sim(self, sim, x, y):
        richtingen = [(0, 1), (1, 0), (0, -1), (-1, 0)]
        random.shuffle(richtingen)
        for dx, dy in richtingen:
            nx, ny = x + dx, y + dy
            if 0 <= nx < GRID_SIZE and 0 <= ny < GRID_SIZE and not self.grid[ny][nx]:
                self.grid[ny][nx] = sim
                self.grid[y][x]   = {}
                return

    def loop_naar_buur(self, sim, x, y, doel_x, doel_y):
        """Beweegt de Sim één stap dichterbij het doel."""
        dx = (doel_x > x) - (doel_x < x)
        dy = (doel_y > y) - (doel_y < y)
        nx, ny = x + dx, y + dy
        if 0 <= nx < GRID_SIZE and 0 <= ny < GRID_SIZE and not self.grid[ny][nx]:
            self.grid[ny][nx] = sim
            self.grid[y][x]   = {}
            return True
        return False

    # UI statusbord

    def update_ui_log(self):
        self.ui_text.delete(1.0, tk.END)
        tekst = "\u2500\u2500\u2500 STATUSBORD \u2500\u2500\u2500\n\n"

        # Nora bovenaan
        nora  = self.nieuwslezer
        zin   = (nora.get("laatste_dialoog") or "...")[:45]
        tekst += f"{nora['icon']} Nora (nieuwslezer)\n"
        tekst += f"   Gevoel:  {nora['stemming']}\n"
        tekst += f"   Zei:     '{zin}'\n\n"

        for sim, _, _ in list(self.vind_sims()):
            dialoog = (sim.get("laatste_dialoog") or "...")[:45]
            tekst  += f"{sim['icon']} {sim['naam']}\n"
            tekst  += f"   Gevoel:  {sim['stemming']}\n"
            tekst  += f"   Honger:  {sim['honger']}/100\n"
            tekst  += f"   Zei:     '{dialoog}'\n\n"

        self.ui_text.insert(tk.END, tekst)

## Deel 5 – Game starten

⚠️ **Let op:** deze cel blokkeert de kernel zolang het venster open is.
Sluit het venster om verder te werken in het notebook.
Zorg dat Ollama actief is (ollama serve of het icoontje in de taakbalk)
voordat je de game start, anders valt elke LLM-beslissing terug op de fallback.

In [ ]:
MijnSimsWereld()

DEBUG nieuws ontvangen: 'Geen NEWS_API_KEY gevonden. Controleer je .env bestand.'
🗞️ Nora las nieuws over 'natuur OR klimaat OR dieren' → voelt zich nieuwsgierig.
🍽️ Emma eet direct uit de buurt!
Emma → eet | behoorlijk hongerig
🍽️ Fatima eet direct uit de buurt!
Fatima → eet | Fatima heeft behoorlijk hongerig en moet eten om haar lichaam te voeden.
🍽️ Lars eet direct uit de buurt!
Lars → eet | Lars is erg hongerig en eten is verstandig
Emma → beweeg | Fallback op basis van vaste spellogica.
Fatima → beweeg | Fallback op basis van vaste spellogica.
Lars → beweeg | Lars is nieuwsgierig en beweegt graag
Emma → rust | Fallback op basis van vaste spellogica. | Variatie: beweeg → rust
Fatima → praat | Om te voorkomen dat Fatima uitgehongerd raakt en om haar lichaam te voeden. | Variatie: beweeg → praat
Lars → rust | Lars is nieuwsgierig en beweegt graag | Variatie: beweeg → rust
🍽️ Emma eet direct uit de buurt!
Emma → eet | Emma deed eet. Behoorlijk hongerig | Ethiek: Voedsel is schaars in 